This file extracts and calibrates the data for the RFQ environment based on real financial data.

Outputs of this file are found in the config.yaml file: 
- sigma_bp = daily yield vol
- sigma = modified duration
- g = LAMBDA * sigma (this is the gap scale that is used to calculate numerical draws)

Instrument of choice: off-the-run 10 year Treasury notes 

In [2]:
import io, requests, pandas as pd
import yfinance as yf

C1:  daily yield vol           sigma_bp     (basis points per day)
C2:  sanity-check it against two other sources
C3:  × modified duration  →    sigma        (price points per day)
     × LAMBDA (0.1)      →     g            (the gap scale)

In [12]:
URL = "https://www.federalreserve.gov/data/yield-curve-tables/feds200628.csv"
raw = requests.get(URL, timeout=60).text.splitlines()

print(len(raw), "lines")
print("\n".join(raw[:10])[:600])

start = next(i for i, l in enumerate(raw) if l.startswith("Date"))
print("header at line", start)

sw = pd.read_csv(io.StringIO("\n".join(raw[start:])),
                  index_col=0, parse_dates=True)

print(gsw.shape)
print([c for c in gsw.columns if c.startswith("SVENY")][:12])


17033 lines
"Note: This is not an official Federal Reserve statistical release. Because this is a staff research product and not an official statistical release, it is subject to delay, revision, or methodological changes without advance notice."

Series,Compounding Convention,Mnemonic(s)
Zero-coupon yield,Continuously Compounded,SVENYXX
Par yield,Coupon-Equivalent,SVENPYXX
Instantaneous forward rate,Continuously Compounded,SVENFXX
One-year forward rate,Coupon-Equivalent,SVEN1FXX
Parameters,N/A,BETA0 to TAU2

Date,BETA0,BETA1,BETA2,BETA3,SVEN1F01,SVEN1F04,SVEN1F09,SVENF01,SVENF02,SVENF03,SVENF04,SVENF05,S
header at line 9
(17023, 99)
['SVENY01', 'SVENY02', 'SVENY03', 'SVENY04', 'SVENY05', 'SVENY06', 'SVENY07', 'SVENY08', 'SVENY09', 'SVENY10', 'SVENY11', 'SVENY12']


In [10]:
#Extract 10 year series and window it 

y10 = pd.to_numeric(gsw["SVENY10"], errors="coerce").dropna()
print("full range:", y10.index.min().date(), "→", y10.index.max().date())
print(y10.tail())

y10w = y10.loc["2025-09-01":]
print(len(y10w), "observations in window")
print("y10 window range:", y10w.index.min().date(), "→", y10w.index.max().date())


full range: 1971-08-16 → 2026-09-11
Date
2026-09-04    4.8154
2026-09-08    4.8370
2026-09-09    4.8465
2026-09-10    4.9673
2026-09-11    4.9735
Name: SVENY10, dtype: float64
258 observations in window
y10 window range: 2025-09-02 → 2026-09-11


In [9]:
#Calculate vol for data calibration

sigma_bp = (y10w * 100).diff().dropna().std()
print(f"sigma_bp = {sigma_bp:.3f} bp/day")

sigma_bp = 4.307 bp/day


In [16]:
# Cross-checking the data (off-run from FRED vs on-run from YFinance)

dgs10 = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS10",
                    index_col=0, parse_dates=True, na_values=".").iloc[:, 0].dropna()

sigma_bp_onrun = (dgs10.loc["2025-09-01":] * 100).diff().dropna().std()
print(f"off-the-run (GSW):  {sigma_bp:.3f} bp/day")
print(f"on-the-run (DGS10): {sigma_bp_onrun:.3f} bp/day")

raw_move = yf.download("^MOVE", start="2025-09-01", progress=False, auto_adjust=True)
close = raw_move["Close"]
if hasattr(close, "columns"):        # newer yfinance returns MultiIndex columns
    close = close.iloc[:, 0]

sigma_bp_implied = float(close.mean()) / 252 ** 0.5
print(f"MOVE mean (annualised): {float(close.mean()):.1f} bp")
print(f"implied daily:          {sigma_bp_implied:.3f} bp/day")

off-the-run (GSW):  4.307 bp/day
on-the-run (DGS10): 4.104 bp/day


$^MOVE: possibly delisted; no price data found  (1d 2025-09-01 -> 2026-09-21)

1 Failed download:
['^MOVE']: possibly delisted; no price data found  (1d 2025-09-01 -> 2026-09-21)


MOVE mean (annualised): nan bp
implied daily:          nan bp/day


In [18]:
# Check correlation between the SVENY10 and DGS10 series 

common = y10w.index.intersection(dgs10.index)
a = (y10.loc[common] * 100).diff().dropna()
b = (dgs10.loc[common] * 100).diff().dropna()

print(f"{len(common)} common dates")
print(f"GSW   sd {a.std():.3f} bp")
print(f"DGS10 sd {b.std():.3f} bp")
print(f"correlation of daily changes: {a.corr(b):.3f}")


258 common dates
GSW   sd 4.307 bp
DGS10 sd 4.094 bp
correlation of daily changes: 0.985


In [26]:
# DV01 calculation 

def mod_duration_par(y, T=10.0):
    """Modified duration of a semiannual par coupon bond. y decimal, T years"""
    D_mod = (1 - (1 + y/2)**(-2*T)) / y
    return D_mod

y_yield = float(y10w.mean()) / 100    
print(f"average 10y yield: {y_now*100:.2f}%")

print(mod_duration_par(0.00001, 10))  # expect ≈ 10.0  (no coupon → duration = maturity)

average 10y yield: 4.38%
9.99947501931775


In [30]:
# Calculating the gap scale, g

maturity_years = 9.75 #first off-the-run: ~1-3 months past issue
LAMBDA = 0.1 #hard coded

mod_duration = mod_duration_par(y_yield, maturity_years)
points_per_bp = mod_duration * 0.0001 * 100
sigma = sigma_bp * points_per_bp
g = LAMBDA * sigma

print(f"mod duration  : {mod_duration:.3f}  ({maturity_years}y at {y_yield*100:.2f}%)")
print(f"points per bp : {points_per_bp:.4f}  (DV01 ${points_per_bp*10_000:,.0f} per $1mm)")
print(f"sigma         : {sigma:.4f} pts/day ({sigma*32:.2f}/32nds)")
print(f"g             : {g:.4f} pts        ({g*32:.2f}/32nds)")
print(f"g on $10mm    : ${g/100*10_000_000:,.0f}")
print(f"ticks per g   : {g/0.001:.1f}")

assert 0.3  <= sigma <= 0.8,  f"GATE FAIL: sigma={sigma:.4f} outside 0.3-0.8"
assert 0.02 <= g     <= 0.10, f"GATE FAIL: g={g:.4f} outside 0.02-0.10"
print("\ngates pass")

mod duration  : 7.865  (9.75y at 4.38%)
points per bp : 0.0786  (DV01 $786 per $1mm)
sigma         : 0.3387 pts/day (10.84/32nds)
g             : 0.0339 pts        (1.08/32nds)
g on $10mm    : $3,387
ticks per g   : 33.9

gates pass


In [32]:
import yaml, numpy as np, datetime

BETA = float(np.log(9) / g)

cfg = {
    "calibration": {
        "source": "GSW SVENY10 (off-the-run fitted curve), daily changes",
        "window_start": "2025-09-01",
        "window_end": str(y10w.index.max().date()),
        "n_obs": int(len(y10w)),
        "sigma_bp": round(float(sigma_bp), 4),
        "maturity_years": float(maturity_years),
        "duration_at_yield_pct": round(float(y_yield) * 100, 3),
        "mod_duration": round(float(mod_duration), 4),
        "sigma_points": round(float(sigma), 8),
        "lambda": float(LAMBDA),
        "g_points": round(float(g), 8),
        "lognormal_sigma": 0.5,
        "beta_rule": "ln(9)/g",
        "beta": round(BETA, 6),
        "disclosure": {"K": 8, "omega": 0.7},
        "notional_mm": 10,
        "computed_on": datetime.date.today().isoformat(),
    },
    "model": {                     # filled at Step 7
        "name": None, "version": None, "provider": None,
        "temperature": 0.5, "reasoning_mode": None,
    },
    "run": {
        "main_seeds": [1, 50],
        "heldout_seeds": [1001, 1020],
        "replicates": 2,
    },
}

with open("config.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, default_flow_style=False)

print(open("config.yaml").read())


calibration:
  source: GSW SVENY10 (off-the-run fitted curve), daily changes
  window_start: '2025-09-01'
  window_end: '2026-09-11'
  n_obs: 258
  sigma_bp: 4.3068
  maturity_years: 9.75
  duration_at_yield_pct: 4.384
  mod_duration: 7.865
  sigma_points: 0.33873104
  lambda: 0.1
  g_points: 0.0338731
  lognormal_sigma: 0.5
  beta_rule: ln(9)/g
  beta: 64.86635
  disclosure:
    K: 8
    omega: 0.7
  notional_mm: 10
  computed_on: '2026-09-22'
model:
  name: null
  version: null
  provider: null
  temperature: 0.5
  reasoning_mode: null
run:
  main_seeds:
  - 1
  - 50
  heldout_seeds:
  - 1001
  - 1020
  replicates: 2

